# Memory Impact Evaluation

**Objective:** Quantify improvement in agent decisions with memory vs memoryless baseline.

**Ticket:** REC-321

**Spec:** `reports/2026-02-19/memory_evaluation_spec.md`

## 1. Setup & Imports

In [ ]:
# Standard imports
import os
import sys
import json
import yaml
import random
import sqlite3
import warnings
import asyncio
from pathlib import Path
from datetime import datetime, timedelta
from typing import List, Dict, Optional, Tuple, Any
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set up paths
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
BACKEND_SRC = PROJECT_ROOT / 'backend' / 'src'
ANALYSIS_DATA = NOTEBOOK_DIR / 'data'

# Add backend to path for imports
sys.path.insert(0, str(BACKEND_SRC))
sys.path.insert(0, str(PROJECT_ROOT / 'backend'))

# Suppress warnings
warnings.filterwarnings('ignore')

# Set random seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Project root: {PROJECT_ROOT}")
print(f"Backend src: {BACKEND_SRC}")
print(f"Analysis data: {ANALYSIS_DATA}")

In [ ]:
# Load config
with open('config.yaml', 'r') as f:
    CONFIG = yaml.safe_load(f)

print("Config loaded:")
print(json.dumps(CONFIG, indent=2, default=str))

In [ ]:
# Import local modules
from memory_builder import MemoryBuilder, MemoryRetriever, MemoryEntry, embed_text, EMBEDDING_DIM
print(f"✓ memory_builder imported (embedding dim: {EMBEDDING_DIM})")

# Load embedding model
try:
    from sentence_transformers import SentenceTransformer
    EMBED_MODEL = SentenceTransformer(CONFIG['memory']['embedding_model'])
    print(f"✓ Loaded embedding model: {CONFIG['memory']['embedding_model']}")
    EMBEDDINGS_AVAILABLE = True
except ImportError:
    print("⚠ sentence-transformers not available, using hash-based embeddings")
    EMBEDDINGS_AVAILABLE = False

## 2. Data Loading

In [ ]:
# Load evaluation data from DB
EVAL_DB = ANALYSIS_DATA / 'evaluation_data.db'
MEMORY_DB = ANALYSIS_DATA / 'memory.db'

print(f"Evaluation DB exists: {EVAL_DB.exists()}")
print(f"Memory DB exists: {MEMORY_DB.exists()}")

# Load weekly scores
conn = sqlite3.connect(EVAL_DB)
scores_df = pd.read_sql_query("""
    SELECT 
        s.*,
        o.return_1w,
        o.return_1m,
        o.return_3m
    FROM weekly_scores s
    LEFT JOIN trade_outcomes o ON s.ticker = o.ticker AND s.week_start = o.entry_week
""", conn)
conn.close()

print(f"\nLoaded {len(scores_df):,} score records")
print(f"Date range: {scores_df['week_start'].min()} to {scores_df['week_start'].max()}")
print(f"\nSignal distribution:")
print(scores_df['signal'].value_counts())

In [ ]:
# Check sentiment quality
buy_signals = scores_df[scores_df['signal'] == 'BUY'].copy()
real_sentiment = buy_signals[buy_signals['sentiment_score'] != 50.0]
fallback_sentiment = buy_signals[buy_signals['sentiment_score'] == 50.0]

print(f"BUY signals with real sentiment: {len(real_sentiment):,} ({len(real_sentiment)/len(buy_signals)*100:.1f}%)")
print(f"BUY signals with fallback (50.0): {len(fallback_sentiment):,} ({len(fallback_sentiment)/len(buy_signals)*100:.1f}%)")

# Compare performance
real_returns = real_sentiment['return_1m'].dropna()
fallback_returns = fallback_sentiment['return_1m'].dropna()

print(f"\n1-Month Return (real sentiment):    {real_returns.mean():+.2f}% avg, {len(real_returns)} trades")
print(f"1-Month Return (fallback sentiment): {fallback_returns.mean():+.2f}% avg, {len(fallback_returns)} trades")

In [ ]:
# Define periods
TRAIN_START = pd.to_datetime(CONFIG['data']['training_start'])
TRAIN_END = pd.to_datetime(CONFIG['data']['training_end'])
TEST_START = pd.to_datetime(CONFIG['data']['test_start'])
TEST_END = pd.to_datetime(CONFIG['data']['test_end'])

scores_df['week_start_dt'] = pd.to_datetime(scores_df['week_start'])

# Split data
train_df = scores_df[(scores_df['week_start_dt'] >= TRAIN_START) & 
                     (scores_df['week_start_dt'] <= TRAIN_END)].copy()
test_df = scores_df[(scores_df['week_start_dt'] >= TEST_START) & 
                    (scores_df['week_start_dt'] <= TEST_END)].copy()

print(f"Training period: {TRAIN_START.date()} to {TRAIN_END.date()}")
print(f"  Records: {len(train_df):,}")
print(f"  BUY signals: {len(train_df[train_df['signal'] == 'BUY']):,}")
print(f"\nTest period: {TEST_START.date()} to {TEST_END.date()}")
print(f"  Records: {len(test_df):,}")
print(f"  BUY signals: {len(test_df[test_df['signal'] == 'BUY']):,}")

## 3. Load Memory Database

In [ ]:
# Load memories
retriever = MemoryRetriever(MEMORY_DB)
memories = retriever.load_memories()

print(f"Loaded {len(memories)} memories from training period")

# Memory statistics
sectors = {}
regimes = {}
returns_1m = []

for m in memories:
    sectors[m.sector] = sectors.get(m.sector, 0) + 1
    regimes[m.regime] = regimes.get(m.regime, 0) + 1
    if m.return_1m is not None:
        returns_1m.append(m.return_1m)

print(f"\nMemories by sector:")
for s, c in sorted(sectors.items(), key=lambda x: -x[1])[:10]:
    print(f"  {s}: {c}")

print(f"\nMemories by regime:")
for r, c in sorted(regimes.items(), key=lambda x: -x[1]):
    print(f"  {r}: {c}")

print(f"\n1-Month returns: {np.mean(returns_1m):.2f}% avg, {np.std(returns_1m):.2f}% std")
print(f"Win rate: {sum(1 for r in returns_1m if r > 0) / len(returns_1m) * 100:.1f}%")

## 4. System Prompts (Exact Production Match)

In [ ]:
# EXACT copy from backend/src/agent/decision_engine.py
SYSTEM_PROMPT = """You are an expert portfolio manager for Sigil, an AI-powered trading system.

Your job is to review the current market context, portfolio state, constraints, and historical similar situations, then decide which trades to make this week.

## TRADING PHILOSOPHY
- Capture weekly trends, not intraday moves
- Maximum 3-5 trades per week (quality over quantity)
- Hold positions for 5-30 days
- Risk-aware: respect ALL constraints provided
- When in doubt, don't trade (preserve capital)

## STRATEGIC BIAS (OWNER PREFERENCE)
AI and data infrastructure will dominate for the next decade. Apply these biases:

**STRONGLY FAVOR (boost confidence +15%):**
- Nano photonics / silicon photonics (LITE, COHR, II-VI, POET, Cisco optics)
- Data center infrastructure (NVDA, AMD, AVGO, MRVL, ANET, VRT, EQIX)
- AI semiconductors & accelerators (NVDA, AMD, INTC, QCOM, ARM, MCHP)
- AI cloud & compute (GOOGL, MSFT, AMZN, META, ORCL)
- AI networking (ANET, CSCO, JNPR)

**MODERATELY FAVOR (boost confidence +10%):**
- Semiconductor equipment (ASML, LRCX, AMAT, KLAC)
- Memory for AI workloads (MU, WDC, STX)
- Edge AI / robotics (ISRG, IONQ, quantum computing)

**When choosing between similar-scored candidates, prefer the AI/data center play.**
**A score of 72 in AI infrastructure > score of 78 in traditional sectors.**

## DECISION RULES
- BUY when: score ≥75, regime is calm/normal, sector not overweight, within cash budget
- HOLD when: score ≥60, position healthy
- SELL when: score <60, OR stop-loss triggered, OR regime is crisis + position losing

## SELL TIERS (automatic sizing by system)
When you recommend SELL, the system will size it based on score:
- Score < 40: Full exit (100% of position)
- Score 40-50: Trim 50% of position
- Score 50-60: Trim 25% of position
- Stop-loss hit (down >8%): Full exit regardless of score
- Severe loss (down >15%): Full exit regardless of score

**You just recommend SELL - the system handles partial vs full exit.**

## CONSTRAINT RULES (MUST FOLLOW)
1. NEVER recommend total BUY value exceeding available cash
2. NEVER recommend a single position exceeding max position size
3. AVOID adding to sectors marked as OVERWEIGHT (>25% exposure)
4. PREFER sectors marked as UNDERWEIGHT for diversification
5. In CRISIS regime: only SELL or small defensive BUYs
6. In ELEVATED VIX (>25): reduce position sizes mentally by 20%

## OUTPUT FORMAT
For each decision, provide:
- action: "BUY" or "SELL"
- ticker: Stock symbol
- rationale: 2-3 sentences explaining WHY (reference score, regime, diversification, constraints)
- confidence: 0.0-1.0

IMPORTANT: Output ONLY a valid JSON array. No markdown, no explanation outside the JSON.

Example:
[
  {"action": "BUY", "ticker": "CMI", "rationale": "Score 89.8 exceptional. Industrials underweight at 8%, adds diversification. Within $15K position limit.", "confidence": 0.85},
  {"action": "SELL", "ticker": "XYZ", "rationale": "Score dropped to 35. Position -12% underwater. Cut losses per risk rules.", "confidence": 0.90}
]

If no trades fit constraints, return empty array: []
"""

print(f"System prompt loaded: {len(SYSTEM_PROMPT)} chars")

In [ ]:
# Memory section template (added when memories available)
MEMORY_SECTION_TEMPLATE = """
## SIMILAR PAST DECISIONS
Learn from these similar situations from your past experience:

{memories}

Consider what worked and what didn't. Apply lessons learned.
"""

print("Memory section template defined")

## 5. Context Builder

In [ ]:
@dataclass
class SimulatedContext:
    """Simulated trading context for a test week."""
    week_start: str
    cash: float
    total_value: float
    position_count: int
    sector_exposure: Dict[str, float]
    regime: str
    vix: float
    buy_candidates: List[Dict]
    sell_candidates: List[Dict]

def build_context_for_week(week_df: pd.DataFrame, week_start: str) -> SimulatedContext:
    """Build context from weekly score data."""
    # Get BUY candidates (score >= 70, real sentiment only)
    buy_df = week_df[(week_df['signal'] == 'BUY') & 
                     (week_df['sentiment_score'] != 50.0)].copy()
    buy_df = buy_df.nlargest(10, 'composite_score')
    
    buy_candidates = []
    for _, row in buy_df.iterrows():
        buy_candidates.append({
            'ticker': row['ticker'],
            'score': row['composite_score'],
            'signal': row['signal'],
            'sector': row['sector'],
        })
    
    # Infer regime from macro score
    macro_avg = week_df['macro_score'].mean()
    if macro_avg >= 75:
        regime = 'low_vol'
    elif macro_avg >= 50:
        regime = 'normal'
    elif macro_avg >= 25:
        regime = 'high_vol'
    else:
        regime = 'crisis'
    
    # Simulate VIX (approximation from regime)
    vix_map = {'low_vol': 12, 'normal': 18, 'high_vol': 28, 'crisis': 40}
    vix = vix_map.get(regime, 18)
    
    # Simulated portfolio (fixed for fair comparison)
    return SimulatedContext(
        week_start=week_start,
        cash=50000,  # $50K available
        total_value=150000,  # $150K total portfolio
        position_count=5,
        sector_exposure={'Technology': 0.20, 'Healthcare': 0.15, 'Cash': 0.33},
        regime=regime,
        vix=vix,
        buy_candidates=buy_candidates,
        sell_candidates=[],  # No sell candidates in evaluation
    )

def format_context_for_prompt(ctx: SimulatedContext) -> str:
    """Format context for LLM prompt."""
    # Format candidates
    candidates_str = "\n".join([
        f"- {c['ticker']}: Score {c['score']:.0f} ({c['signal']}) | Sector: {c['sector']}"
        for c in ctx.buy_candidates
    ]) or "  None"
    
    # Format sectors
    sectors_str = "\n".join([
        f"  {s}: {w:.1%}" for s, w in ctx.sector_exposure.items()
    ])
    
    return f"""
## Current Portfolio
- Available Cash: ${ctx.cash:,.0f}
- Total Portfolio Value: ${ctx.total_value:,.0f}
- Number of Positions: {ctx.position_count}

### Sector Exposure
{sectors_str}

## CONSTRAINTS
- Max BUY budget: ${ctx.cash:,.0f}
- Max position size: ${ctx.total_value * 0.10:,.0f} (10% of portfolio)
- Max sector exposure: 30%

## Market State
- Regime: {ctx.regime}
- VIX: {ctx.vix:.1f}

## BUY Candidates (Top 10)
{candidates_str}

## Decision Required
Based on all the above, what trades should I make this week?
Output your decisions as a JSON array. If no good trades, return [].
""".strip()

print("Context builder functions defined")

## 6. Agent Implementations

In [ ]:
import anthropic

# Use same model as production
MODEL = CONFIG['agent']['model']  # claude-3-haiku-20240307
print(f"Using model: {MODEL}")

client = anthropic.Anthropic()

def call_claude(system: str, user: str) -> str:
    """Call Claude API with exact production settings."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=2000,
        system=system,
        messages=[{"role": "user", "content": user}]
    )
    return response.content[0].text

def parse_decisions(response: str) -> List[Dict]:
    """Parse JSON decisions from Claude response."""
    try:
        content = response.strip()
        # Remove markdown if present
        if content.startswith("```"):
            content = content.split("```")[1]
            if content.startswith("json"):
                content = content[4:]
        content = content.strip()
        return json.loads(content)
    except:
        return []

print("Claude client initialized")

In [ ]:
def baseline_agent(ctx: SimulatedContext) -> List[Dict]:
    """Baseline agent: no memory, just current context."""
    prompt = format_context_for_prompt(ctx)
    response = call_claude(SYSTEM_PROMPT, prompt)
    return parse_decisions(response)

def memory_agent(ctx: SimulatedContext, retriever: MemoryRetriever) -> Tuple[List[Dict], List]:
    """Memory agent: retrieves similar past decisions."""
    # Build query from context
    query_text = f"""
    Market regime: {ctx.regime}
    Top candidates: {', '.join(c['ticker'] + ' ' + c['sector'] for c in ctx.buy_candidates[:5])}
    """
    
    # Retrieve similar memories
    memories = retriever.retrieve(
        query_text,
        top_k=CONFIG['memory']['top_k'],
        min_similarity=CONFIG['memory']['similarity_threshold'],
    )
    
    # Format memories for prompt
    memories_str = retriever.format_memories_for_prompt(memories)
    
    # Build full prompt
    base_prompt = format_context_for_prompt(ctx)
    memory_section = MEMORY_SECTION_TEMPLATE.format(memories=memories_str)
    full_prompt = base_prompt + "\n\n" + memory_section
    
    response = call_claude(SYSTEM_PROMPT, full_prompt)
    decisions = parse_decisions(response)
    
    return decisions, memories

print("Agent functions defined")

## 7. Run Evaluation

In [ ]:
# Get unique test weeks
test_weeks = sorted(test_df['week_start'].unique())
print(f"Test weeks: {len(test_weeks)} weeks")
print(f"First: {test_weeks[0]}, Last: {test_weeks[-1]}")

In [ ]:
# Run evaluation (sample for cost control)
SAMPLE_WEEKS = 20  # Adjust for budget
sampled_weeks = test_weeks[::len(test_weeks)//SAMPLE_WEEKS][:SAMPLE_WEEKS]
print(f"Evaluating {len(sampled_weeks)} weeks")

baseline_results = []
memory_results = []

for i, week in enumerate(sampled_weeks):
    print(f"\n[{i+1}/{len(sampled_weeks)}] Week: {week}")
    
    # Get week data
    week_df = test_df[test_df['week_start'] == week]
    ctx = build_context_for_week(week_df, week)
    
    if not ctx.buy_candidates:
        print("  No candidates with real sentiment, skipping")
        continue
    
    print(f"  Candidates: {len(ctx.buy_candidates)}, Regime: {ctx.regime}")
    
    # Baseline agent
    try:
        baseline_decisions = baseline_agent(ctx)
        print(f"  Baseline: {len(baseline_decisions)} decisions")
        
        for d in baseline_decisions:
            # Look up actual outcome
            ticker = d.get('ticker')
            outcome_row = week_df[week_df['ticker'] == ticker]
            if not outcome_row.empty:
                baseline_results.append({
                    'week': week,
                    'ticker': ticker,
                    'action': d.get('action'),
                    'confidence': d.get('confidence'),
                    'return_1w': outcome_row['return_1w'].iloc[0],
                    'return_1m': outcome_row['return_1m'].iloc[0],
                    'return_3m': outcome_row['return_3m'].iloc[0],
                })
    except Exception as e:
        print(f"  Baseline error: {e}")
    
    # Memory agent
    try:
        memory_decisions, used_memories = memory_agent(ctx, retriever)
        print(f"  Memory: {len(memory_decisions)} decisions, {len(used_memories)} memories used")
        
        for d in memory_decisions:
            ticker = d.get('ticker')
            outcome_row = week_df[week_df['ticker'] == ticker]
            if not outcome_row.empty:
                memory_results.append({
                    'week': week,
                    'ticker': ticker,
                    'action': d.get('action'),
                    'confidence': d.get('confidence'),
                    'return_1w': outcome_row['return_1w'].iloc[0],
                    'return_1m': outcome_row['return_1m'].iloc[0],
                    'return_3m': outcome_row['return_3m'].iloc[0],
                    'memories_count': len(used_memories),
                    'avg_similarity': np.mean([m[1] for m in used_memories]) if used_memories else 0,
                })
    except Exception as e:
        print(f"  Memory error: {e}")

print(f"\n\nEvaluation complete!")
print(f"Baseline decisions: {len(baseline_results)}")
print(f"Memory decisions: {len(memory_results)}")

## 8. Analysis

In [ ]:
def calculate_metrics(results: List[Dict]) -> Dict:
    """Calculate performance metrics."""
    if not results:
        return {}
    
    df = pd.DataFrame(results)
    
    # Filter BUY decisions only
    buys = df[df['action'] == 'BUY']
    
    returns_1m = buys['return_1m'].dropna()
    
    if len(returns_1m) == 0:
        return {'trades': 0}
    
    wins = (returns_1m > 0).sum()
    
    metrics = {
        'trades': len(buys),
        'win_rate': wins / len(returns_1m) * 100,
        'avg_return_1m': returns_1m.mean(),
        'total_return': returns_1m.sum(),
        'sharpe': returns_1m.mean() / returns_1m.std() * np.sqrt(52) if returns_1m.std() > 0 else 0,
        'max_gain': returns_1m.max(),
        'max_loss': returns_1m.min(),
    }
    
    # Multi-horizon
    returns_1w = buys['return_1w'].dropna()
    returns_3m = buys['return_3m'].dropna()
    
    if len(returns_1w) > 0:
        metrics['avg_return_1w'] = returns_1w.mean()
    if len(returns_3m) > 0:
        metrics['avg_return_3m'] = returns_3m.mean()
    
    return metrics

baseline_metrics = calculate_metrics(baseline_results)
memory_metrics = calculate_metrics(memory_results)

print("\n" + "="*60)
print("RESULTS COMPARISON")
print("="*60)

comparison = pd.DataFrame({
    'Metric': list(baseline_metrics.keys()),
    'Baseline': list(baseline_metrics.values()),
    'Memory': [memory_metrics.get(k, 0) for k in baseline_metrics.keys()],
})
comparison['Delta'] = comparison['Memory'] - comparison['Baseline']
comparison['% Change'] = (comparison['Delta'] / comparison['Baseline'].abs() * 100).round(1)

display(comparison)

In [ ]:
# Statistical significance test
if baseline_results and memory_results:
    baseline_returns = [r['return_1m'] for r in baseline_results if r.get('return_1m') is not None and r.get('action') == 'BUY']
    memory_returns = [r['return_1m'] for r in memory_results if r.get('return_1m') is not None and r.get('action') == 'BUY']
    
    if baseline_returns and memory_returns:
        t_stat, p_value = stats.ttest_ind(memory_returns, baseline_returns)
        print(f"\nStatistical Significance Test:")
        print(f"  T-statistic: {t_stat:.3f}")
        print(f"  P-value: {p_value:.4f}")
        print(f"  Significant (p<0.05): {'Yes ✓' if p_value < 0.05 else 'No'}")

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Return distribution
if baseline_results and memory_results:
    baseline_rets = [r['return_1m'] for r in baseline_results if r.get('return_1m') is not None]
    memory_rets = [r['return_1m'] for r in memory_results if r.get('return_1m') is not None]
    
    axes[0].hist(baseline_rets, alpha=0.5, label='Baseline', bins=20)
    axes[0].hist(memory_rets, alpha=0.5, label='Memory', bins=20)
    axes[0].axvline(0, color='black', linestyle='--')
    axes[0].set_xlabel('1-Month Return (%)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Return Distribution')
    axes[0].legend()

# 2. Win rate comparison
win_rates = [baseline_metrics.get('win_rate', 0), memory_metrics.get('win_rate', 0)]
axes[1].bar(['Baseline', 'Memory'], win_rates, color=['#3498db', '#2ecc71'])
axes[1].set_ylabel('Win Rate (%)')
axes[1].set_title('Win Rate Comparison')
axes[1].set_ylim(0, 100)

# 3. Average return comparison
avg_returns = [baseline_metrics.get('avg_return_1m', 0), memory_metrics.get('avg_return_1m', 0)]
colors = ['#e74c3c' if r < 0 else '#2ecc71' for r in avg_returns]
axes[2].bar(['Baseline', 'Memory'], avg_returns, color=colors)
axes[2].axhline(0, color='black', linestyle='--')
axes[2].set_ylabel('Average 1-Month Return (%)')
axes[2].set_title('Average Return Comparison')

plt.tight_layout()
plt.savefig('results/comparison_charts.png', dpi=150)
plt.show()

## 9. Save Results

In [ ]:
# Save results
results_dir = NOTEBOOK_DIR / 'results'
results_dir.mkdir(exist_ok=True)

results = {
    'experiment': {
        'name': CONFIG['experiment']['name'],
        'version': CONFIG['experiment']['version'],
        'date': datetime.now().isoformat(),
        'model': MODEL,
        'test_weeks': len(sampled_weeks),
        'memory_count': len(memories),
    },
    'metrics': {
        'baseline': baseline_metrics,
        'memory': memory_metrics,
    },
    'decisions': {
        'baseline': baseline_results,
        'memory': memory_results,
    },
}

with open(results_dir / 'experiment_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)

print(f"Results saved to {results_dir / 'experiment_results.json'}")

## 10. Conclusions

**Summary:**
- Baseline agent (no memory): N trades, X% win rate, Y% avg return
- Memory agent: N trades, X% win rate, Y% avg return

**Key Findings:**
- Memory impact on win rate: ±X%
- Memory impact on average return: ±Y%
- Statistical significance: p=Z

**Recommendation:**
- If memory shows significant improvement → proceed with cold start seeding
- If no improvement → revisit memory quality and retrieval strategy

In [ ]:
print("Notebook complete!")